# Task 4 — Notebook 01: Feature Engineering from CDL History

**Purpose (NAFSI Task 4):** Crop-type prediction (corn, soybean, third class) using **multi-year CDL** as features/labels, optionally **NDVI** and **SMAP**. This notebook loads **processed wide Parquet** stacks (same information as interim NetCDF, produced by `scripts/process_interim_to_parquet.py`).

**Inputs (processed):**
- `data/processed/cdl/cdl_stack_wide.parquet` + `cdl_stack_spatial_metadata.json`
- `data/processed/ndvi/ndvi_weekly_{year}_wide.parquet` (+ `*_metadata.json` per year)
- `data/processed/smap/smap_weekly_{year}_wide.parquet` (+ `*_metadata.json` per year)
- `data/processed/task2/rotation_metrics.parquet` — when Task 2 produces it, join rotation scores here

**Outputs:** Objects `cdl_stack`, `ndvi_all`, `smap_all` as **xarray** `DataArray`s (equivalent layout to the old interim loaders). NDVI/SMAP years with fewer weeks are NaN-padded in `time` to a common length so stacks can concatenate.

In [5]:
# NOTE: This notebook was scaffolded with AI assistance.

import sys
from pathlib import Path

import pandas as pd
import yaml

_cwd = Path.cwd().resolve()
REPO_ROOT = next(
    (p for p in (_cwd, *_cwd.parents) if (p / "requirements.txt").is_file() and (p / "src").is_dir()),
    None,
)
if REPO_ROOT is None:
    raise RuntimeError("Could not find repo root. cd to the repo and retry.")
sys.path.insert(0, str(REPO_ROOT))

from src.io.processed_loaders import (
    load_cdl_stack_from_processed,
    load_ndvi_weekly_all_years_processed,
    load_smap_weekly_all_years_processed,
)

with open(REPO_ROOT / "configs" / "task4_crop_mapping.yaml") as f:
    cfg = yaml.safe_load(f)

# Rolling panel years live under ``panel`` (see DEC-007).
py = cfg["panel"]
train_span = py["train_years"]
test_y = int(py["test_year"])
val_y = int(py["val_year"])
print("Panel train years (inclusive):", train_span, "| val:", val_y, "| test:", test_y)

Panel train years (inclusive): [2013, 2022] | val: 2022 | test: 2023


In [6]:
cdl_stack = load_cdl_stack_from_processed(REPO_ROOT)
ndvi_all = load_ndvi_weekly_all_years_processed(REPO_ROOT)
smap_all = load_smap_weekly_all_years_processed(REPO_ROOT)

print("cdl_stack", dict(cdl_stack.sizes))
print("ndvi_all", dict(ndvi_all.sizes))
print("smap_all", dict(smap_all.sizes))

rot_path = REPO_ROOT / "data" / "processed" / "task2" / "rotation_metrics.parquet"
if rot_path.is_file():
    rotation_scores = pd.read_parquet(rot_path)
    print("Loaded rotation metrics:", rotation_scores.shape)
else:
    rotation_scores = None
    print("No rotation_metrics.parquet yet — run Task 2 or skip rotation feature.")

# Next: encode sequences, transitions, 3×3 composition; labels from cdl_stack.sel(year=test_y).


cdl_stack {'year': 18, 'y': 1520, 'x': 2048}
ndvi_all {'calendar_year': 26, 'time': 26, 'y': 1520, 'x': 2048}
smap_all {'calendar_year': 11, 'time': 53, 'y': 1520, 'x': 2048}
No rotation_metrics.parquet yet — run Task 2 or skip rotation feature.
